# Prompt 13: Importing Existing Infrastructure and Inspecting State

**Terraform Associate (004) — Objective 7: Maintaining Infrastructure with Terraform**

This notebook covers how to bring existing infrastructure under Terraform management using the `import` block and the legacy CLI import command, how to inspect state and outputs with CLI commands, and how to enable verbose logging for troubleshooting.

---

## Topics Covered

1. Why Import Is Needed
2. The `import` Block (Terraform 1.5+ — Preferred)
3. CLI Import Command (Legacy)
4. When to Use Each Approach
5. Inspecting State with CLI Commands
6. Verbose Logging with `TF_LOG`
7. Exam-Style Questions
8. Real-World Scenarios

---

## 1. Why Import Is Needed

Terraform manages infrastructure by tracking resources in a **state file**. If infrastructure was created:

- Manually via a cloud console
- By another tool (CloudFormation, Pulumi, scripts)
- Before your team adopted Terraform

…then Terraform has no knowledge of it. Without importing, Terraform would try to **create duplicate resources** or simply ignore the existing ones, leading to drift and conflicts.

**Import** associates an existing real-world resource with a Terraform resource address in state so Terraform can manage it going forward.

### Key Exam Point

| Situation | What Happens Without Import |
|---|---|
| Resource exists in cloud, not in state | Terraform tries to create it again on next apply |
| Resource is in config but not state | `terraform plan` shows it as a new resource to create |
| After import | Terraform tracks the resource; subsequent plans show no changes |

> **Rule of thumb**: Before Terraform can manage an existing resource, it must be in state.

---

## 2. The `import` Block (Terraform 1.5+ — Preferred Method)

Introduced in **Terraform 1.5**, the `import` block is the **preferred** way to import resources. It integrates with `terraform plan` and supports automatic configuration generation.

### Syntax

```hcl
import {
  to = aws_instance.web
  id = "i-0abcd1234ef567890"
}
```

| Argument | Purpose |
|---|---|
| `to` | The Terraform resource address the real resource will be mapped to |
| `id` | The cloud provider's unique ID for the resource (format varies by resource type) |

### Workflow

```
1. Add import block to config
        ↓
2. terraform plan          ← shows the import action and optionally generates HCL
        ↓
3. terraform apply         ← executes import, writes resource to state
        ↓
4. Remove import block     ← no longer needed after successful import
```

### `generate config_for` — Automatic HCL Generation

If you haven't written the resource configuration yet, Terraform can generate it:

```bash
terraform plan -generate-config-out=generated.tf
```

This writes a complete resource block to `generated.tf` based on the real infrastructure's attributes. You can then review, edit, and incorporate it into your configuration.

> **Key Exam Points**:
> - `import` block shows in `terraform plan` output — you can review before applying
> - Works with `for_each` to import multiple resources at once (Terraform 1.7+)
> - No separate `terraform import` command needed
> - `import` block can be removed after apply — it's only needed once

In [ ]:
# Example: import block for an existing EC2 instance
# File: main.tf

import_block_example = '''
# Step 1: Declare the import block
import {
  to = aws_instance.web
  id = "i-0abcd1234ef567890"
}

# Step 2: Declare the matching resource block
# (or use -generate-config-out to have Terraform write this for you)
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  tags = {
    Name = "web-server"
  }
}
'''

print(import_block_example)

In [ ]:
# Example: terraform plan output when import block is present

plan_output_example = '''
$ terraform plan

Terraform will perform the following actions:

  # aws_instance.web will be imported
    resource "aws_instance" "web" {
        ami           = "ami-0c55b159cbfafe1f0"
        id            = "i-0abcd1234ef567890"
        instance_type = "t3.micro"
        tags          = {
            "Name" = "web-server"
        }
        # ... other attributes
    }

Plan: 1 to import, 0 to add, 0 to change, 0 to destroy.
'''

print(plan_output_example)

In [ ]:
# Example: generating configuration automatically

generate_config_example = '''
# Only the import block is needed — no resource block yet
import {
  to = aws_s3_bucket.data
  id = "my-existing-bucket-name"
}
'''

cli_generate = '''
# Run this to have Terraform write the resource block for you:
$ terraform plan -generate-config-out=generated_resources.tf

# Terraform writes generated_resources.tf with:
resource "aws_s3_bucket" "data" {
  bucket = "my-existing-bucket-name"
  # ... all discovered attributes
}

# Review generated_resources.tf, then run:
$ terraform apply
'''

print(generate_config_example)
print(cli_generate)

---

## 3. CLI Import Command (Legacy Approach)

Before Terraform 1.5, the only way to import was with the `terraform import` CLI command:

```bash
terraform import <resource_address> <resource_id>
```

### Key Differences from the `import` Block

| Aspect | `import` Block | CLI `terraform import` |
|---|---|---|
| HCL config generation | Yes (`-generate-config-out`) | **No** — must write manually |
| Visible in `plan` | Yes | No — runs immediately |
| Reviewable before apply | Yes | No |
| Idempotent / version-controlled | Yes (in `.tf` file) | No (imperative command) |
| Terraform version | 1.5+ | All versions |
| Still supported | N/A | Yes |

### CLI Import Workflow

```
1. Write the resource block in config manually
        ↓
2. Run: terraform import aws_instance.web i-0abcd1234ef567890
        ↓
3. Resource is added to state immediately
        ↓
4. Run terraform plan to verify no unintended changes
```

> **Critical Exam Point**: The CLI `terraform import` command does **NOT** generate HCL configuration. You must write the resource block yourself before importing, otherwise Terraform will not know how to manage the resource.

In [ ]:
# CLI import command examples

cli_import_examples = '''
# Import an EC2 instance
$ terraform import aws_instance.web i-0abcd1234ef567890

# Import an S3 bucket
$ terraform import aws_s3_bucket.data my-existing-bucket

# Import an Azure resource group
$ terraform import azurerm_resource_group.example /subscriptions/<sub>/resourceGroups/myRG

# Import a resource in a module
$ terraform import module.vpc.aws_vpc.main vpc-0a1b2c3d4e5f

# Import a resource using count index
$ terraform import 'aws_instance.servers[0]' i-0abcd1234ef567890

# Import a resource using for_each key
$ terraform import 'aws_instance.servers["web"]' i-0abcd1234ef567890
'''

print(cli_import_examples)

---

## 4. When to Use Each Approach

| Scenario | Recommended Approach |
|---|---|
| Terraform 1.5+ and you want config generated | `import` block with `-generate-config-out` |
| Terraform 1.5+ and you have config already | `import` block (shows in plan, version-controlled) |
| Older Terraform version (< 1.5) | CLI `terraform import` |
| Quick one-off import in CI/CD pipeline | CLI `terraform import` |
| Batch import of many resources | `import` block with `for_each` (Terraform 1.7+) |
| Team collaboration and auditability | `import` block (lives in `.tf` files, visible in PR) |

### Summary Decision Tree

```
Is Terraform >= 1.5?
  YES → Use import block (preferred)
  NO  → Use CLI terraform import (only option)

Do you have the HCL config written?
  YES → Import block or CLI import (both work)
  NO  → Import block + -generate-config-out (block only)
```

> **Exam tip**: The `import` block is described as **preferred** in HashiCorp documentation. The CLI command is still fully supported but is the older, manual approach.

---

## 5. Inspecting State with CLI Commands

Once resources are in state, Terraform provides several commands to inspect what it knows about your infrastructure.

### `terraform state list`

Lists all resource addresses tracked in state.

```bash
terraform state list               # all resources
terraform state list aws_instance  # filter by resource type prefix
terraform state list module.vpc    # filter to a specific module
```

### `terraform state show <address>`

Shows all attributes Terraform has recorded for a specific resource.

```bash
terraform state show aws_instance.web
terraform state show module.vpc.aws_subnet.public[0]
```

### `terraform show`

Shows a human-readable representation of the **entire state file** or a **saved plan file**.

```bash
terraform show           # human-readable state
terraform show -json     # JSON format state
terraform show plan.out  # contents of a saved plan file
```

### `terraform output`

Shows values of output variables defined in the configuration.

```bash
terraform output                     # show all outputs
terraform output instance_ip         # show specific output
terraform output -json               # JSON format (all outputs)
terraform output -json instance_ip   # JSON format (specific)
terraform output -raw instance_ip    # raw string value (no quotes)
```

### `terraform console`

Opens an interactive REPL for evaluating Terraform expressions against current state.

```bash
terraform console
```

Use cases:
- Test built-in functions: `> upper("hello")`
- Inspect variable values: `> var.region`
- Evaluate complex expressions: `> [for s in var.subnets : cidrsubnet(s, 4, 1)]`
- Check resource attributes: `> aws_instance.web.public_ip`

### Command Summary Table

| Command | What It Shows | Notes |
|---|---|---|
| `terraform state list` | All resource addresses in state | Supports prefix filtering |
| `terraform state show <addr>` | All attributes of one resource | Full details from state |
| `terraform show` | Human-readable view of state or plan | Use `-json` for machine-readable |
| `terraform output` | Output variable values | Use `-json` or `-raw` for scripting |
| `terraform console` | Interactive expression evaluator | Read-only; does not modify state |

In [ ]:
# Example: terraform state list output

state_list_output = '''
$ terraform state list

aws_instance.web
aws_security_group.allow_http
aws_s3_bucket.logs
module.vpc.aws_vpc.main
module.vpc.aws_subnet.public[0]
module.vpc.aws_subnet.public[1]
module.vpc.aws_internet_gateway.gw

# Filter to a module:
$ terraform state list module.vpc

module.vpc.aws_vpc.main
module.vpc.aws_subnet.public[0]
module.vpc.aws_subnet.public[1]
module.vpc.aws_internet_gateway.gw
'''

print(state_list_output)

In [ ]:
# Example: terraform state show output

state_show_output = '''
$ terraform state show aws_instance.web

# aws_instance.web:
resource "aws_instance" "web" {
    ami                          = "ami-0c55b159cbfafe1f0"
    arn                          = "arn:aws:ec2:us-east-1:123456789012:instance/i-0abcd1234ef567890"
    associate_public_ip_address  = true
    availability_zone            = "us-east-1a"
    id                           = "i-0abcd1234ef567890"
    instance_state               = "running"
    instance_type                = "t3.micro"
    private_ip                   = "10.0.1.15"
    public_ip                    = "54.23.198.44"
    tags                         = {
        "Name" = "web-server"
    }
    vpc_security_group_ids       = [
        "sg-0a1b2c3d4e5f",
    ]
    # ... additional computed attributes
}
'''

print(state_show_output)

In [ ]:
# Example: terraform output commands

output_examples = '''
# In configuration:
output "instance_ip" {
  value = aws_instance.web.public_ip
}

output "bucket_name" {
  value = aws_s3_bucket.logs.id
}

# CLI usage:
$ terraform output
bucket_name = "my-logs-bucket-20240101"
instance_ip = "54.23.198.44"

$ terraform output instance_ip
"54.23.198.44"

$ terraform output -raw instance_ip
54.23.198.44

$ terraform output -json
{
  "bucket_name": {
    "sensitive": false,
    "type": "string",
    "value": "my-logs-bucket-20240101"
  },
  "instance_ip": {
    "sensitive": false,
    "type": "string",
    "value": "54.23.198.44"
  }
}
'''

print(output_examples)

In [ ]:
# Example: terraform console interactive session

console_example = '''
$ terraform console
> upper("hello world")
"HELLO WORLD"

> length(["a", "b", "c"])
3

> cidrsubnet("10.0.0.0/16", 8, 1)
"10.0.1.0/24"

> aws_instance.web.public_ip
"54.23.198.44"

> var.environment
"production"

> [for s in ["us-east-1a", "us-east-1b"] : upper(s)]
[
  "US-EAST-1A",
  "US-EAST-1B",
]

> exit   # or Ctrl+D to quit
'''

print(console_example)

---

## 6. Verbose Logging with `TF_LOG`

Terraform uses environment variables to control logging verbosity. This is essential for debugging provider issues, authentication failures, and unexpected plan results.

### `TF_LOG` — Log Level

Set this variable to enable logging. Log levels from most verbose to least:

| Level | Use Case |
|---|---|
| `TRACE` | Maximum detail — all provider API calls, HTTP requests/responses |
| `DEBUG` | Provider plugin communication, useful for provider issues |
| `INFO` | General informational messages |
| `WARN` | Warnings that may indicate configuration problems |
| `ERROR` | Error messages only |
| _(unset)_ | No logging output |

### `TF_LOG_PATH` — Write Logs to File

```bash
export TF_LOG=TRACE
export TF_LOG_PATH=./terraform.log
terraform apply
```

Logs are written to `terraform.log` instead of stderr.

### `TF_LOG_CORE` and `TF_LOG_PROVIDER` — Granular Control

Available since Terraform 0.15:

| Variable | Controls |
|---|---|
| `TF_LOG_CORE` | Terraform core engine logs |
| `TF_LOG_PROVIDER` | Provider plugin logs |

```bash
# Only verbose provider logs, quiet core logs
export TF_LOG_CORE=ERROR
export TF_LOG_PROVIDER=TRACE
terraform apply

# Only verbose core, quiet provider
export TF_LOG_CORE=TRACE
export TF_LOG_PROVIDER=INFO
terraform plan
```

### When to Use Each Level

| Scenario | Recommended Level |
|---|---|
| Provider authentication failure | `DEBUG` or `TRACE` |
| API rate limiting or 429 errors | `DEBUG` |
| Unexpected resource creation/deletion | `DEBUG` |
| Full HTTP request/response dump | `TRACE` |
| CI/CD pipeline troubleshooting | `INFO` or `DEBUG` |
| Normal operations | Unset (no logging) |

> **Exam tip**: `TF_LOG=TRACE` shows ALL provider API calls including authentication tokens. Never commit log files to version control — they may contain credentials.

In [ ]:
# TF_LOG usage examples

tf_log_examples = '''
# Enable TRACE logging to stderr
$ TF_LOG=TRACE terraform apply

# Enable TRACE logging and write to file
$ TF_LOG=TRACE TF_LOG_PATH=./terraform-debug.log terraform apply

# On Windows (PowerShell):
$env:TF_LOG = "TRACE"
$env:TF_LOG_PATH = ".\\terraform-debug.log"
terraform apply

# On Windows (cmd.exe):
set TF_LOG=TRACE
set TF_LOG_PATH=.\\terraform-debug.log
terraform apply

# Separate core and provider log levels:
$ TF_LOG_CORE=INFO TF_LOG_PROVIDER=TRACE terraform plan

# Disable logging (unset the variable):
$ unset TF_LOG
$ unset TF_LOG_PATH
'''

tf_log_trace_output = '''
# Sample TRACE output excerpt (provider API call):
2024-01-15T10:22:31.123Z [TRACE] provider.terraform-provider-aws: 2024/01/15 10:22:31 [DEBUG] AWS Auth provider chain
2024-01-15T10:22:31.456Z [TRACE] provider.terraform-provider-aws: AWSClient: Sending HTTP Request:
2024-01-15T10:22:31.456Z [TRACE]   GET https://ec2.us-east-1.amazonaws.com/?Action=DescribeInstances&...
2024-01-15T10:22:31.789Z [TRACE] provider.terraform-provider-aws: AWSClient: Received HTTP Response: 200
'''

print(tf_log_examples)
print(tf_log_trace_output)

---

## 7. Exam-Style Questions

### Question 1

A team has been running AWS EC2 instances manually for two years and now wants to manage them with Terraform. They are using Terraform 1.6. Which approach should they use to bring the instances under Terraform management?

**A.** Run `terraform import aws_instance.web <id>` directly — this also generates the HCL configuration  
**B.** Add `import` blocks to configuration, run `terraform plan -generate-config-out=generated.tf`, review the output, then run `terraform apply`  
**C.** Copy the instance's ARN into a `terraform.tfstate` file manually  
**D.** Run `terraform refresh` — this detects and imports existing infrastructure automatically  

<details><summary>Answer</summary>

**B** is correct. With Terraform 1.6, the `import` block is the preferred method. Using `-generate-config-out` lets Terraform write the HCL resource configuration automatically based on the real resource's attributes. After reviewing the generated config, `terraform apply` imports the resource into state.

**A** is wrong because the CLI `terraform import` command does NOT generate HCL — you must write the resource block yourself first.

**C** is wrong and dangerous — never manually edit the state file.

**D** is wrong — `terraform refresh` updates state to match existing infrastructure but does NOT import resources that have never been in state.

</details>

---

### Question 2

A developer wants to see all attributes that Terraform has stored for `aws_security_group.allow_http`. Which command should they run?

**A.** `terraform state list aws_security_group.allow_http`  
**B.** `terraform output aws_security_group.allow_http`  
**C.** `terraform state show aws_security_group.allow_http`  
**D.** `terraform show aws_security_group.allow_http`  

<details><summary>Answer</summary>

**C** is correct. `terraform state show <address>` displays all attributes Terraform has stored in state for a specific resource — this includes computed attributes like IDs, ARNs, and other values populated after creation.

**A** is wrong — `terraform state list` shows resource addresses (names), not attributes. It accepts a prefix filter but doesn't show attributes.

**B** is wrong — `terraform output` shows declared `output` blocks, not individual resource attributes.

**D** is wrong — `terraform show` without an address shows the entire state or a plan file; it does not accept a resource address as an argument.

</details>

---

### Question 3

A Terraform apply is failing with a cryptic provider error. A DevOps engineer wants to see the full HTTP requests and responses being sent to the AWS API to diagnose the issue. Which environment variable setting should they use?

**A.** `TF_LOG=ERROR`  
**B.** `TF_LOG=WARN`  
**C.** `TF_LOG=DEBUG`  
**D.** `TF_LOG=TRACE`  

<details><summary>Answer</summary>

**D** is correct. `TF_LOG=TRACE` is the most verbose level and shows ALL provider API calls including full HTTP requests and responses. This is the appropriate level when you need to see exactly what API calls Terraform is making.

**C** (`DEBUG`) is useful for provider plugin communication but may not show full HTTP body content.

**A** and **B** only show errors and warnings respectively — not helpful for diagnosing API communication issues.

Note: `TF_LOG_PATH=./terraform.log` should also be set so the voluminous TRACE output is captured to a file rather than flooding the terminal.

</details>

---

## 8. Real-World Scenarios

<details><summary>Scenario 1: Adopting a Manually Created RDS Database</summary>

**Situation**: A startup has been running an Amazon RDS MySQL database that was created manually in the AWS console 18 months ago. The team is now adopting Terraform for all infrastructure and needs to bring this database under Terraform management without deleting and recreating it.

**HCL Configuration**:

```hcl
# Step 1: Add the import block
import {
  to = aws_db_instance.main
  id = "myapp-production-db"  # The RDS DB identifier
}

# Step 2: Terraform generates this with -generate-config-out
# (or write manually, matching actual values)
resource "aws_db_instance" "main" {
  identifier        = "myapp-production-db"
  engine            = "mysql"
  engine_version    = "8.0.32"
  instance_class    = "db.t3.medium"
  allocated_storage = 100
  db_name           = "myappdb"
  username          = "admin"
  # password is write-only — must supply but won't force replacement
  password          = var.db_password

  skip_final_snapshot = false

  tags = {
    Environment = "production"
  }
}
```

**CLI Commands**:

```bash
# Generate config automatically
terraform plan -generate-config-out=rds_generated.tf

# Review rds_generated.tf, fill in write-only values (password)
# Then apply
terraform apply

# Verify no drift after import
terraform plan  # Should show: No changes.
```

**Expected Outcome**: The RDS instance is now tracked in Terraform state. Subsequent `terraform plan` runs will show no changes if the config matches reality. The database is NOT recreated — it is adopted in place.

**Exam Sub-Objective**: Import existing resources into Terraform state (Objective 7).

</details>

---

<details><summary>Scenario 2: Debugging Provider Authentication with TF_LOG</summary>

**Situation**: A team's CI/CD pipeline is failing on `terraform plan` with a generic `NoCredentialProviders: no valid providers in chain` error when using AWS. The engineer suspects a role assumption issue but the error message is not specific enough.

**CLI Commands**:

```bash
# Enable TRACE logging to capture full auth chain and API calls
export TF_LOG=TRACE
export TF_LOG_PATH=./auth-debug.log

terraform plan

# Review the log file
grep -A 5 "AWS Auth" auth-debug.log
grep "AssumeRole" auth-debug.log
grep "HTTP Response" auth-debug.log | head -20
```

**Sample Log Excerpt** (what the engineer finds in auth-debug.log):

```
[TRACE] provider.terraform-provider-aws: Attempting to assume role arn:aws:iam::123456789012:role/TerraformRole
[TRACE] provider.terraform-provider-aws: HTTP Response: 403 Forbidden
[TRACE] provider.terraform-provider-aws: AccessDenied: User is not authorized to perform sts:AssumeRole
```

**Resolution**: The IAM policy on the CI/CD execution role is missing the `sts:AssumeRole` permission for the target role. The engineer adds the permission, and the pipeline succeeds.

```bash
# Clean up after debugging
unset TF_LOG
unset TF_LOG_PATH
rm auth-debug.log  # Log may contain credentials — delete when done
```

**Expected Outcome**: TRACE logging reveals the exact API call and error response, turning a cryptic error into a clear IAM permission issue.

**Exam Sub-Objective**: Use `TF_LOG` and `TF_LOG_PATH` to enable verbose logging (Objective 7).

</details>

---

<details><summary>Scenario 3: Auditing Resources After a Terraform Apply</summary>

**Situation**: An infrastructure team just ran `terraform apply` that created 12 new resources including VPC subnets, security groups, and EC2 instances. A junior engineer wants to audit what was created and verify key attributes (like private IPs and security group IDs) without going into the AWS console.

**CLI Commands**:

```bash
# See everything that was created
terraform state list

# Output:
# aws_instance.app[0]
# aws_instance.app[1]
# aws_security_group.app
# aws_security_group.alb
# module.vpc.aws_vpc.main
# module.vpc.aws_subnet.private[0]
# module.vpc.aws_subnet.private[1]
# module.vpc.aws_subnet.public[0]
# module.vpc.aws_subnet.public[1]
# aws_lb.app
# aws_lb_target_group.app
# aws_lb_listener.http

# Inspect a specific EC2 instance
terraform state show 'aws_instance.app[0]'

# Get the load balancer DNS name
terraform output alb_dns_name

# Get all outputs in JSON for scripting
terraform output -json > outputs.json

# Test an expression in console
terraform console
> aws_lb.app.dns_name
"my-app-lb-1234567890.us-east-1.elb.amazonaws.com"
> exit
```

**Expected Outcome**: The engineer can verify all created resources, their IDs, IPs, and DNS names without leaving the terminal or touching the AWS console.

**Exam Sub-Objective**: Use `terraform state list`, `terraform state show`, `terraform output`, and `terraform console` to inspect infrastructure (Objective 7).

</details>

---

<details><summary>Scenario 4: Importing Multiple Existing S3 Buckets (Legacy CLI Method)</summary>

**Situation**: A company has 4 existing S3 buckets used for application logs, backups, assets, and deployments. They were created before Terraform adoption (Terraform 1.3 — before `import` blocks). The team needs to import all 4 buckets into their Terraform configuration.

**HCL Configuration** (written manually first — required for CLI import):

```hcl
resource "aws_s3_bucket" "logs" {
  bucket = "mycompany-app-logs-prod"
}

resource "aws_s3_bucket" "backups" {
  bucket = "mycompany-backups-prod"
}

resource "aws_s3_bucket" "assets" {
  bucket = "mycompany-static-assets"
}

resource "aws_s3_bucket" "deployments" {
  bucket = "mycompany-deploy-artifacts"
}
```

**CLI Commands** (import each bucket individually):

```bash
# Must write resource blocks BEFORE running these commands
terraform import aws_s3_bucket.logs mycompany-app-logs-prod
terraform import aws_s3_bucket.backups mycompany-backups-prod
terraform import aws_s3_bucket.assets mycompany-static-assets
terraform import aws_s3_bucket.deployments mycompany-deploy-artifacts

# Verify all 4 are now in state
terraform state list

# Check for drift — plan should show minimal/no changes
terraform plan
```

**Note**: If `terraform plan` shows changes after import (e.g., versioning settings, lifecycle rules), update the HCL config to match the real bucket settings before applying.

**Expected Outcome**: All 4 buckets are now in state. Terraform can manage their configuration going forward without recreating them.

**Exam Sub-Objective**: Use `terraform import` CLI command to import existing resources; understand that config must be written manually first (Objective 7).

</details>

---

<details><summary>Scenario 5: Using terraform console to Debug a Complex Expression</summary>

**Situation**: A network engineer is writing Terraform to create 6 subnets from a /16 VPC CIDR. They want to use `cidrsubnet()` but are unsure of the correct `newbits` and `netnum` values to get /24 subnets. They also need to verify the output of a `for` expression that filters the subnet list before testing it in a real apply.

**HCL Configuration** (work in progress):

```hcl
variable "vpc_cidr" {
  default = "10.0.0.0/16"
}

variable "subnet_count" {
  default = 6
}

# Not yet sure if this expression is correct...
locals {
  subnet_cidrs = [for i in range(var.subnet_count) : cidrsubnet(var.vpc_cidr, 8, i)]
}
```

**CLI Commands** (testing before apply):

```bash
$ terraform console

# Test cidrsubnet to understand the math
> cidrsubnet("10.0.0.0/16", 8, 0)
"10.0.0.0/24"

> cidrsubnet("10.0.0.0/16", 8, 1)
"10.0.1.0/24"

> cidrsubnet("10.0.0.0/16", 8, 5)
"10.0.5.0/24"

# Test the full for expression
> [for i in range(6) : cidrsubnet("10.0.0.0/16", 8, i)]
[
  "10.0.0.0/24",
  "10.0.1.0/24",
  "10.0.2.0/24",
  "10.0.3.0/24",
  "10.0.4.0/24",
  "10.0.5.0/24",
]

# Verify actual local value with current variable values
> local.subnet_cidrs
[
  "10.0.0.0/24",
  "10.0.1.0/24",
  "10.0.2.0/24",
  "10.0.3.0/24",
  "10.0.4.0/24",
  "10.0.5.0/24",
]

> exit
```

**Expected Outcome**: The engineer confirms their expression produces the correct CIDR blocks before running `terraform plan` or `terraform apply`. `terraform console` provides instant feedback with no infrastructure changes.

**Exam Sub-Objective**: Use `terraform console` for interactive expression evaluation and function testing (Objective 7).

</details>

---

## Quick Reference Summary

### Import Methods

| Feature | `import` Block (1.5+) | `terraform import` CLI |
|---|---|---|
| Generates HCL config | Yes (`-generate-config-out`) | No |
| Visible in plan | Yes | No |
| Version-controlled | Yes (in .tf files) | No |
| Terraform version | 1.5+ | All |
| HashiCorp preferred | **Yes** | Legacy |

### State Inspection Commands

| Command | What It Does |
|---|---|
| `terraform state list` | List all resource addresses in state |
| `terraform state show <addr>` | Show all attributes of one resource |
| `terraform show` | Human-readable view of state or plan file |
| `terraform output` | Show declared output values |
| `terraform console` | Interactive expression REPL (read-only) |

### Logging Variables

| Variable | Purpose |
|---|---|
| `TF_LOG` | Master log level (TRACE/DEBUG/INFO/WARN/ERROR) |
| `TF_LOG_PATH` | File to write logs to |
| `TF_LOG_CORE` | Log level for Terraform core only |
| `TF_LOG_PROVIDER` | Log level for provider plugins only |

### Log Levels (Most to Least Verbose)

```
TRACE > DEBUG > INFO > WARN > ERROR
```

> Use `TRACE` to see all API calls. Use `DEBUG` for provider issues. Always set `TF_LOG_PATH` to avoid log flooding the terminal.